In [1]:
import os
os.environ["JAX_MPS_ASYNC_DISPATCH"]="1"

In [2]:
import jax

from jax import numpy as jnp

import numpy as np

import scipy

from matplotlib import pyplot as plt
from matplotlib import patches as patches
from matplotlib import cm

import copy
import time
import datetime

In [3]:
key = jax.random.key(0)

dtype = jnp.float32

Platform 'mps' is experimental and not all JAX functionality may be correctly supported!
[jax-mps] JAX_MPS_ASYNC_DISPATCH is ON: async dispatch enabled — may be much faster, but experimental and may break. Unset it for safe synchronous execution.


In [4]:
T_R     = 4e5  # temperature of random agent (K)
T_C     = 2e6  # temperature of causal agent (K)
TAU     = 10.  # simulation time horizon (seconds)
EPSILON = .025 # timesteps (seconds)

NUM_ROLLOUTS = 100_000

SUBSAMPLING = True
NUM_SUBSAMPLES = 10

KDE_CHUNK_SIZE = 100

In [5]:
M     = 1e-21                                # mass (kg)
L     = 400                                  # length (meters) 
Q_MIN = jnp.array([0. , 0.  ], dtype=dtype)  # minimum box displacements
Q_MAX = jnp.array([L  , L/5.], dtype=dtype)  # maximum box displacements
P_MAX = M * jnp.abs(Q_MAX - Q_MIN) / EPSILON # maximum momentum

In [6]:
def step_p(f):
    new_p = jnp.clip(f * EPSILON, -P_MAX, P_MAX)
    return new_p

def step_q(q, p, new_p):
    new_q = q  +  .5 * EPSILON * (p + new_p) / M
    return new_q

def enforce_elastic_collisions(p, q):
    reflect_min = q < Q_MIN
    reflect_max = q > Q_MAX
    q = jnp.where(reflect_min, 2 * Q_MIN - q, q)
    q = jnp.where(reflect_max, 2 * Q_MAX - q, q)
    p = jnp.where(reflect_min | reflect_max, -p, p)
    return p, q

def step_phase_space(p, q, f):
    new_p        = step_p(f)
    new_q        = step_q(q, p, new_p)
    new_p, new_q = enforce_elastic_collisions(new_p, new_q)
    return new_p, new_q

In [7]:
timesteps = int(TAU / EPSILON)

In [8]:
scale = np.sqrt(M * scipy.constants.Boltzmann * T_R) / EPSILON

In [9]:
def generate_fs(key):
    # scale calculation formula from https://math.stackexchange.com/a/1426406
    noise  = jax.random.normal(key, shape=(NUM_ROLLOUTS, 2), dtype=dtype)
    noise *= scale
    return noise

def rollouts(p, q, key):
    """
    all_qs.shape = (timesteps, num_samples, q_dim)
    """
    ps = jnp.tile(p[None, :], (NUM_ROLLOUTS, 1))
    qs = jnp.tile(q[None, :], (NUM_ROLLOUTS, 1))

    all_qs = copy.deepcopy(qs[None, :])

    key, subkey = jax.random.split(key)
    fs          = generate_fs(subkey)
    first_fs    = copy.deepcopy(fs)
    
    for i in range(timesteps):
        ps, qs      = step_phase_space(ps, qs, fs)
        all_qs      = jnp.append(all_qs, qs[None, :], axis=0)
        key, subkey = jax.random.split(key)
        fs          = generate_fs(subkey)

    return all_qs, first_fs

In [10]:
# The following code is modified from 
# [1]: https://github.com/scipy/scipy/blob/main/scipy/stats/_kde.py
# [2]: https://github.com/scipy/scipy/blob/main/scipy/stats/_stats.pyx#L744
# [3]: code generated by Google Gemini Flash 3.6 to implement gaussian kde
# with diagonal bandwidth matrix (the idea to use a diagonal kernel was also
# suggested by Google Gemini Flash 3.6)
# 
# [1] has a copyright notice which I have reproduced below
# 
# -------------------------------------------------------------------------------
#
#  Define classes for (uni/multi)-variate kernel density estimation.
#
#  Currently, only Gaussian kernels are implemented.
#
#  Written by: Robert Kern
#
#  Date: 2004-08-09
#
#  Modified: 2005-02-10 by Robert Kern.
#              Contributed to SciPy
#            2005-10-07 by Robert Kern.
#              Some fixes to match the new scipy_core
#
#  Copyright 2004-2005 by Enthought, Inc.
#
# -------------------------------------------------------------------------------


def diag_gaussian_kde_logpdfs(dataset):
    d, m = dataset.shape
    
    std_per_dim = jnp.std(dataset, axis=1, keepdims=True)
    scotts_factor = jnp.power(m, -1. / (d + 4))
    diag_bandwidth = std_per_dim * scotts_factor  # shape (d, 1)
    dataset_ = (dataset / diag_bandwidth).T.astype(dtype)

    log_pdfs = jnp.zeros(m)
    for j in range(0, m, KDE_CHUNK_SIZE):
        arg = jnp.sum((dataset_ - dataset_[j : j+KDE_CHUNK_SIZE][:, None]) ** 2., axis=-1)
        logits = jax.scipy.special.logsumexp(-arg/2., axis=1)
        log_pdfs = log_pdfs.at[j : j+KDE_CHUNK_SIZE].set(logits)
    
    return log_pdfs

In [11]:
def log_vol_fracs(dataset):
    if SUBSAMPLING:
        # subsampling idea from Google Gemini to speed up KDE
        frame_select = int(timesteps / NUM_SUBSAMPLES)
        dataset = dataset[1::frame_select]
    dataset = dataset.transpose((2, 0, 1)).reshape(-1, NUM_ROLLOUTS).astype(dtype)
    
    log_pdfs = diag_gaussian_kde_logpdfs(dataset)
    log_omega = -log_pdfs
    log_volume_fracs = log_omega - jax.scipy.special.logsumexp(log_omega)
    return log_volume_fracs

In [12]:
def entropic_force(p, q, key):
    paths, fs = rollouts(p, q, key)
    return jnp.mean( fs * log_vol_fracs(paths)[:, None] , axis=0 )

In [13]:
@jax.jit
def step_macrostate(p, q, key):
    key, subkey = jax.random.split(key)
    f_c         = 2. * T_C / T_R * entropic_force(p, q, subkey)
    p, q        = step_phase_space(p, q, f_c)
    return p, q, key

In [ ]:
p = jnp.array([0.   , 0.   ])
q = jnp.array([L/10., L/10.])

path = copy.deepcopy(q[None, :])

start_time = time.time()
for timestep in range(100):
    elapsed_time = time.time() - start_time
    print(timestep, datetime.timedelta(seconds=int(elapsed_time)), q)

    p, q, key = step_macrostate(p, q, key)
    path = jnp.append(path, q[None, :], axis=0)

0 0:00:00 [40. 40.]
1 0:00:10 [40.086723 40.02475 ]
2 0:00:18 [40.149605 40.30607 ]
3 0:00:25 [40.23497 40.76589]
4 0:00:33 [39.98851  40.744278]
5 0:00:41 [39.623516 40.353127]
6 0:00:49 [39.59686 40.27367]
7 0:00:57 [39.104317 40.982513]
8 0:01:05 [38.126034 41.55114 ]
9 0:01:13 [37.296024 42.059925]
10 0:01:21 [37.017956 42.94949 ]
11 0:01:29 [37.12626  43.372963]
12 0:01:36 [36.975735 43.609726]
13 0:01:44 [36.95187  43.929394]
14 0:01:52 [36.73528  44.173496]
15 0:02:00 [36.702168 44.372574]
16 0:02:08 [37.37917 44.36723]
17 0:02:15 [37.87785  44.150295]
18 0:02:23 [37.657295 44.0802  ]
19 0:02:31 [36.767113 43.99451 ]
20 0:02:39 [36.53648  43.834503]
21 0:02:47 [37.295444 43.350445]
22 0:02:55 [37.981606 42.478825]
23 0:03:03 [38.20904 41.49585]
24 0:03:11 [38.440144 40.588844]
25 0:03:19 [38.940136 40.537712]
26 0:03:27 [39.289074 40.71103 ]
27 0:03:35 [39.302258 40.146183]
28 0:03:43 [39.759216 39.610214]
29 0:03:51 [40.291584 39.243614]
30 0:03:59 [40.952534 38.94532 ]
31 0:04

In [ ]:
fig, ax = plt.subplots(figsize=(13, 13))
ax.scatter(*path[::10].T)
ax.set_xlim(Q_MIN[0], Q_MAX[0])
ax.set_ylim(Q_MIN[1], Q_MAX[1])
ax.set(aspect='equal')
plt.show()